In [43]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append("../src")

from preprocessing import clean_text, create_sentiment

In [44]:
file_path = "C:\\Users\\ASUS\\Desktop\\mobile-review-sentiment-analysis\\data\\raw\\amazon_mobile_reviews.csv"

df = pd.read_csv(file_path)



In [45]:
df.shape

(20000, 9)

In [46]:
df.head()

,review_id,product_id,reviewer_id,review_text,review_title,rating,helpful_votes,review_date,sentiment
0,1,120401325X,A30TL5EWN6DFXT,They look good and stick good! I just don't li...,Looks Good,4.0,"[0, 0]","05 21, 2014",Positive
1,2,120401325X,ASY55RVNIL0UD,These stickers work like the review says they ...,Really great product.,5.0,"[0, 0]","01 14, 2014",Positive
2,3,120401325X,A2TMXE2AFO7ONB,These are awesome and make my phone look so st...,LOVE LOVE LOVE,5.0,"[0, 0]","06 26, 2014",Positive
3,4,120401325X,AWJ0WZQYMYFQ4,Item arrived in great time and was in perfect ...,Cute!,4.0,"[4, 4]","10 21, 2013",Positive
4,5,120401325X,ATX7CZYFXI1KW,"awesome! stays on, and looks great. can be use...",leopard home button sticker for iphone 4s,5.0,"[2, 3]","02 3, 2013",Positive


In [47]:
df.columns.tolist()

['review_id',
 'product_id',
 'reviewer_id',
 'review_text',
 'review_title',
 'rating',
 'helpful_votes',
 'review_date',
 'sentiment']

In [48]:
required_columns = [
    "review_id",
    "product_id",
    "reviewer_id",
    "review_text",
    "review_title",
    "rating",
    "helpful_votes",
    "review_date"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are available.")

All required columns are available.


In [49]:
df = df.dropna(subset=["review_text"])

df["review_text"] = df["review_text"].astype(str)

print("Shape after removing missing reviews:", df.shape)

Shape after removing missing reviews: (19987, 9)


In [50]:
df.duplicated().sum()

np.int64(0)

In [51]:
df["rating"] = pd.to_numeric(
    df["rating"],
    errors="coerce"
)

df = df.dropna(subset=["rating"])

df["rating"] = df["rating"].astype(int)

print(df["rating"].value_counts().sort_index())

rating
1     1931
2     1427
3     2289
4     4025
5    10315
Name: count, dtype: int64


In [52]:
df["sentiment"] = df["rating"].apply(create_sentiment)

df[
    ["rating", "sentiment"]
].head(20)

,rating,sentiment
0,4,Positive
1,5,Positive
2,5,Positive
3,4,Positive
4,5,Positive
5,3,Neutral
6,5,Positive
7,1,Negative
8,5,Positive
9,5,Positive


In [53]:
print(
    df[
        ["rating", "sentiment"]
    ].drop_duplicates().sort_values("rating")
)

    rating sentiment
7        1  Negative
26       2  Negative
5        3   Neutral
0        4  Positive
1        5  Positive


In [54]:
df["review_title"] = df["review_title"].fillna("")

df["full_text"] = (
    df["review_title"].astype(str)
    + " "
    + df["review_text"].astype(str)
)

df[[
    "review_title",
    "review_text",
    "full_text",
    "rating",
    "sentiment"
]].head()

,review_title,review_text,full_text,rating,sentiment
0,Looks Good,They look good and stick good! I just don't li...,Looks Good They look good and stick good! I ju...,4,Positive
1,Really great product.,These stickers work like the review says they ...,Really great product. These stickers work like...,5,Positive
2,LOVE LOVE LOVE,These are awesome and make my phone look so st...,LOVE LOVE LOVE These are awesome and make my p...,5,Positive
3,Cute!,Item arrived in great time and was in perfect ...,Cute! Item arrived in great time and was in pe...,4,Positive
4,leopard home button sticker for iphone 4s,"awesome! stays on, and looks great. can be use...",leopard home button sticker for iphone 4s awes...,5,Positive


In [55]:
df["clean_text"] = df["full_text"].apply(clean_text)


In [56]:
df[[
    "full_text",
    "clean_text"
]].head(10)

,full_text,clean_text
0,Looks Good They look good and stick good! I ju...,look good look good stick good like rounded sh...
1,Really great product. These stickers work like...,really great product sticker work like review ...
2,LOVE LOVE LOVE These are awesome and make my p...,love love love awesome make phone look stylish...
3,Cute! Item arrived in great time and was in pe...,cute item arrived great time perfect condition...
4,leopard home button sticker for iphone 4s awes...,leopard home button sticker iphone awesome sta...
5,Cute These make using the home button easy. My...,cute make using home button easy daughter like...
6,best thing ever.. Came just as described.. It ...,best thing ever came described come unstuck cu...
7,not a good Idea it worked for the first week t...,good idea worked first week charge phone waste...
8,"Solid Case Good case, solid build. Protects ph...",solid case good case solid build protects phon...
9,Perfect Case This is a fantastic case. Very st...,perfect case fantastic case stylish protects p...


In [57]:
df = df[df["clean_text"].str.strip() != ""]
print(df.head())

   review_id  product_id     reviewer_id  \
0          1  120401325X  A30TL5EWN6DFXT   
1          2  120401325X   ASY55RVNIL0UD   
2          3  120401325X  A2TMXE2AFO7ONB   
3          4  120401325X   AWJ0WZQYMYFQ4   
4          5  120401325X   ATX7CZYFXI1KW   

                                         review_text  \
0  They look good and stick good! I just don't li...   
1  These stickers work like the review says they ...   
2  These are awesome and make my phone look so st...   
3  Item arrived in great time and was in perfect ...   
4  awesome! stays on, and looks great. can be use...   

                                review_title  rating helpful_votes  \
0                                 Looks Good       4        [0, 0]   
1                      Really great product.       5        [0, 0]   
2                             LOVE LOVE LOVE       5        [0, 0]   
3                                      Cute!       4        [4, 4]   
4  leopard home button sticker for iphone 4s    

In [58]:
print(df["sentiment"].value_counts())

sentiment
Positive    14338
Negative     3358
Neutral      2289
Name: count, dtype: int64


In [59]:
print(df["sentiment"].value_counts(normalize=True)*100)

sentiment
Positive    71.743808
Negative    16.802602
Neutral     11.453590
Name: proportion, dtype: float64


In [60]:
output_path = "C:/Users/ASUS/Desktop/mobile-review-sentiment-analysis/data/interim/1_reviews_cleaned.csv"

df.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: C:/Users/ASUS/Desktop/mobile-review-sentiment-analysis/data/interim/1_reviews_cleaned.csv
